# Yemen MarketShield AI — Walk-Forward Validation (Research Only)

**Purpose:** strengthen the evaluation of the experimental next-month wheat-price-increase flag using monthly walk-forward replay. This notebook does not establish operational readiness.

## Important limitations

- The target is a **greater-than-5% rise in the next month's `c_wheat` close estimate**, not a measured food crisis or a validated humanitarian outcome.
- The World Bank Real-Time Prices series mixes direct price data and model-estimated missing values, and its historical estimates can be updated. This notebook uses the frozen release dated `2026-03-24`; historical revisions create a possible vintage/point-in-time limitation.
- The target period 2022 onward was already viewed in earlier experiments. Therefore, these results are a corrected **retrospective walk-forward replay**, not a completely untouched independent test. A genuine final test requires independent observed prices or a future dataset vintage that was not used during model development.
- Rows with missing, non-finite, or non-positive current/next-month target prices are excluded from supervised evaluation instead of being silently labelled as non-events.
- The included prediction CSVs are cached per-origin results. Their market/month keys, target labels, target changes, and momentum-baseline values are checked against the loaded source data before reuse; incompatible caches trigger a refit. Set `RETRAIN_MODELS = True` to force a refit.
- Do not use this model for real-world humanitarian, purchasing, financial, or policy decisions. Any future use requires an independent evaluation, data freshness controls, probability/threshold calibration, and human review.

Sources: [World Bank food prices](https://microdata.worldbank.org/catalog/4508), [exchange rates](https://microdata.worldbank.org/catalog/6159), [energy prices](https://microdata.worldbank.org/catalog/6133).

## 1. Install / import dependencies

The notebook installs LightGBM only if it is unavailable.

In [1]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("lightgbm") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "lightgbm"])

import io
import zipfile
import os
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score,
    balanced_accuracy_score, confusion_matrix
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)


## 2. Load the same three World Bank ZIP files

In Google Colab, upload the three source archives listed below. For a quick reproduction of the included metric tables, upload the two prediction-cache CSVs from this validation package in the same dialog. If the caches are absent, the notebook fits the two LightGBM variants at every forecast origin and may take several minutes. In local Jupyter, put all available files beside the notebook.

The stored per-origin prediction caches are derived outputs from earlier runs of the same prequential model configurations. The notebook checks their keys, target labels, target changes, and momentum-baseline values against the source-derived rows before reuse; an incompatible cache is rejected and both models are refit. They are not independent ground truth.

In [2]:
EXPECTED = {
    "food": "YEM_RTFP_mkt_2009_2026-03-24.csv",
    "fx": "YEM_RTFX_mkt_2009_2026-03-24.csv",
    "energy": "YEM_RTEP_mkt_2009_2026-03-24.csv",
}
EXPECTED_ZIPS = [
    "YEM_RTFP_mkt_2009_2026-03-24.zip",
    "YEM_RTFX_mkt_2009_2026-03-24.zip",
    "YEM_RTEP_mkt_2009_2026-03-24.zip",
]

try:
    from google.colab import files
    print("Upload the three World Bank ZIP archives listed in the project README.")
    uploaded = files.upload()
    # Save selected archives and optional cached prediction CSVs in the working directory.
    for uploaded_name, uploaded_contents in uploaded.items():
        Path(uploaded_name).write_bytes(uploaded_contents)
except ImportError:
    from pathlib import Path
    missing = [name for name in EXPECTED_ZIPS if not Path(name).is_file()]
    if missing:
        raise FileNotFoundError(
            "Missing source ZIP files: " + ", ".join(missing) +
            ". Download the version 2026-03-24 from the official source links."
        )
    uploaded = {name: Path(name).read_bytes() for name in EXPECTED_ZIPS}

def read_csv_from_uploaded_zip(expected_csv):
    matches = []
    for filename, contents in uploaded.items():
        if filename.lower().endswith(".zip"):
            with zipfile.ZipFile(io.BytesIO(contents)) as zf:
                candidates = [n for n in zf.namelist()
                              if os.path.basename(n) == expected_csv]
                for name in candidates:
                    with zf.open(name) as f:
                        matches.append(pd.read_csv(f, low_memory=False))
    if len(matches) != 1:
        raise ValueError(
            f"Expected exactly one ZIP containing {expected_csv}; found {len(matches)}"
        )
    return matches[0]

food = read_csv_from_uploaded_zip(EXPECTED["food"])
fx = read_csv_from_uploaded_zip(EXPECTED["fx"])
energy = read_csv_from_uploaded_zip(EXPECTED["energy"])
for frame in (food, fx, energy):
    frame["price_date"] = pd.to_datetime(frame["price_date"])

print("Food:", food.shape, "| FX:", fx.shape, "| Energy:", energy.shape)


Food: (5175, 165) | FX: (5175, 40) | Energy: (5175, 34)


## 3. Validate keys and merge model inputs

`Market Average` is excluded from modelling so the aggregate is not counted as a separate local market.

In [3]:
KEY = ["geo_id", "price_date"]
for name, frame in [("food", food), ("fx", fx), ("energy", energy)]:
    if frame.duplicated(KEY).any():
        raise ValueError(f"Duplicate market-month keys detected in {name}.")

food_keys = set(map(tuple, food[KEY].astype(str).to_numpy()))
fx_keys = set(map(tuple, fx[KEY].astype(str).to_numpy()))
energy_keys = set(map(tuple, energy[KEY].astype(str).to_numpy()))
print("Food = FX keys:", food_keys == fx_keys)
print("Food = Energy keys:", food_keys == energy_keys)
print("Shared key count:", len(food_keys & fx_keys & energy_keys))

food_cols = [
    "geo_id", "price_date", "mkt_name", "adm1_name", "lat", "lon", "currency",
    "wheat", "c_wheat", "c_wheat_flour", "c_rice", "c_sugar", "c_food_price_index",
    "trust_wheat", "trust_wheat_flour", "trust_rice", "trust_sugar",
    "trust_food_price_index", "data_coverage", "data_coverage_recent",
    "index_confidence_score", "spatially_interpolated"
]
fx_cols = [
    "geo_id", "price_date", "exchange_rate_unofficial", "c_exchange_rate_unofficial",
    "inflation_exchange_rate_unofficial", "trust_exchange_rate_unofficial",
    "c_milling_cost_wheat", "c_wage_non_qualified_labour", "c_wage_qualified_labour"
]
energy_cols = [
    "geo_id", "price_date", "fuel_diesel", "c_fuel_diesel", "trust_fuel_diesel",
    "fuel_gas", "c_fuel_gas", "trust_fuel_gas",
    "fuel_petrol_gasoline", "c_fuel_petrol_gasoline", "trust_fuel_petrol_gasoline",
    "spatially_interpolated"
]
food_selected = food[[c for c in food_cols if c in food.columns]].copy()
food_selected = food_selected.rename(columns={
    "spatially_interpolated": "spatially_interpolated_food"
})
fx_selected = fx[[c for c in fx_cols if c in fx.columns]].copy()
energy_selected = energy[[c for c in energy_cols if c in energy.columns]].copy()
energy_selected = energy_selected.rename(columns={
    "spatially_interpolated": "spatially_interpolated_energy"
})

merged = food_selected.merge(
    fx_selected, on=KEY, how="inner", validate="one_to_one"
).merge(
    energy_selected, on=KEY, how="inner", validate="one_to_one"
)
print("Merged shape:", merged.shape)
print("Markets including aggregate:", merged["mkt_name"].nunique())
print("Period:", merged["price_date"].min().date(), "to", merged["price_date"].max().date())
print("Missing key target values:", merged[["c_wheat", "c_exchange_rate_unofficial", "c_fuel_diesel"]].isna().sum().to_dict())


Food = FX keys: True
Food = Energy keys: True
Shared key count: 5175
Merged shape: (5175, 39)
Markets including aggregate: 25
Period: 2009-01-01 to 2026-03-01
Missing key target values: {'c_wheat': 0, 'c_exchange_rate_unofficial': 0, 'c_fuel_diesel': 0}


## 4. Define the forecasting task and build features

At each market-month origin `t`, the model predicts whether `c_wheat` at the **next consecutive month** will be more than 5% higher than the current month. The model is retrained at each forecast origin. Training examples are admitted only when their target month is at or before the current origin date, preventing a future label from crossing a time boundary.

The input price and confidence fields still come from the same retrospective source snapshot; this procedure does not eliminate World Bank vintage revisions.

In [4]:
df = merged[merged["mkt_name"].ne("Market Average")].copy()
df = df.sort_values(["geo_id", "price_date"]).reset_index(drop=True)

price_cols = [
    "c_wheat", "c_wheat_flour", "c_rice", "c_sugar", "c_food_price_index",
    "c_exchange_rate_unofficial", "c_milling_cost_wheat", "c_fuel_diesel",
    "c_fuel_gas", "c_fuel_petrol_gasoline"
]
price_cols = [c for c in price_cols if c in df.columns]
for col in price_cols:
    for lag in (1, 2, 3, 12):
        df[f"{col}_lag{lag}"] = df.groupby("geo_id")[col].shift(lag)

for col in [
    "c_wheat", "c_wheat_flour", "c_rice", "c_exchange_rate_unofficial",
    "c_fuel_diesel", "c_fuel_gas", "c_fuel_petrol_gasoline"
]:
    if col in df.columns:
        for lag in (1, 3, 12):
            df[f"{col}_log_change_{lag}"] = np.log(
                df[col].clip(lower=1e-6) / df[f"{col}_lag{lag}"].clip(lower=1e-6)
            )

quality_cols = [
    "trust_wheat", "trust_wheat_flour", "trust_rice", "trust_sugar",
    "trust_food_price_index", "trust_exchange_rate_unofficial",
    "trust_fuel_diesel", "trust_fuel_gas", "trust_fuel_petrol_gasoline",
    "data_coverage", "data_coverage_recent", "index_confidence_score",
    "spatially_interpolated_food", "spatially_interpolated_energy"
]
quality_cols = [c for c in quality_cols if c in df.columns]
for col in quality_cols:
    df[f"{col}_lag1"] = df.groupby("geo_id")[col].shift(1)

df["month_num"] = df["price_date"].dt.month
df["month_sin"] = np.sin(2 * np.pi * df["month_num"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month_num"] / 12)
df["next_c_wheat"] = df.groupby("geo_id")["c_wheat"].shift(-1)
df["next_date"] = df.groupby("geo_id")["price_date"].shift(-1)
month_gap = (
    (df["next_date"].dt.year - df["price_date"].dt.year) * 12
    + (df["next_date"].dt.month - df["price_date"].dt.month)
)
df = df[month_gap.eq(1)].copy()

# Build the supervised target only where both monthly wheat prices are valid.
# Missing/invalid outcomes must not be silently converted into negative labels.
df["c_wheat"] = pd.to_numeric(df["c_wheat"], errors="coerce")
df["next_c_wheat"] = pd.to_numeric(df["next_c_wheat"], errors="coerce")
valid_target = (
    df["c_wheat"].gt(0)
    & df["next_c_wheat"].gt(0)
    & np.isfinite(df["c_wheat"])
    & np.isfinite(df["next_c_wheat"])
)
invalid_target_rows = int((~valid_target).sum())
df = df.loc[valid_target].copy()

df["next_pct_change"] = df["next_c_wheat"] / df["c_wheat"] - 1
valid_target_change = np.isfinite(df["next_pct_change"])
invalid_change_rows = int((~valid_target_change).sum())
df = df.loc[valid_target_change].copy()
df["shock_gt5"] = (df["next_pct_change"] > 0.05).astype("int8")

# Leave the momentum baseline undefined when its previous price is invalid.
valid_previous_price = (
    df["c_wheat_lag1"].gt(0) & np.isfinite(df["c_wheat_lag1"])
)
df["previous_month_change"] = np.nan
df.loc[valid_previous_price, "previous_month_change"] = (
    df.loc[valid_previous_price, "c_wheat"]
    / df.loc[valid_previous_price, "c_wheat_lag1"] - 1
)
print(f"Excluded rows with invalid current/next wheat prices: {invalid_target_rows}")
print(f"Excluded rows with invalid calculated target changes: {invalid_change_rows}")

core_numeric = list(dict.fromkeys(
    price_cols
    + [f"{c}_lag{lag}" for c in price_cols for lag in (1, 2, 3, 12)
       if f"{c}_lag{lag}" in df.columns]
    + [c for c in df.columns if "_log_change_" in c]
    + ["month_sin", "month_cos"]
))
quality_numeric = quality_cols + [f"{c}_lag1" for c in quality_cols if f"{c}_lag1" in df.columns]
quality_numeric = list(dict.fromkeys(quality_numeric))
enhanced_numeric = core_numeric + [c for c in quality_numeric if c not in core_numeric]
print("Local-market rows for prediction:", len(df))
print("Target positive rate:", round(df["shock_gt5"].mean(), 3))
print("Core numeric features:", len(core_numeric), "| Enhanced numeric features:", len(enhanced_numeric))


Excluded rows with invalid current/next wheat prices: 0
Excluded rows with invalid calculated target changes: 0
Local-market rows for prediction: 4944
Target positive rate: 0.183
Core numeric features: 73 | Enhanced numeric features: 101


## 5. Monthly walk-forward replay

For each month, train only on prior examples whose next-month outcome is already known. The threshold is selected on the 2019–2021 validation period by maximum F1, then fixed for the 2022–March 2026 replay.

In [5]:
# Monthly retraining function plus guarded reuse of saved predictions.
# Set RETRAIN_MODELS = True to force both LightGBM variants to be refit.
# Otherwise, saved caches are reused only when their rows and targets match the
# source-derived dataset exactly. Incompatible/missing caches cause an automatic refit.

RETRAIN_MODELS = False
FORECAST_START = pd.Timestamp("2018-12-01")
FORECAST_END = pd.Timestamp("2026-02-01")
CACHE_KEY_COLUMNS = ["geo_id", "price_date", "next_date"]
CACHE_CHECK_COLUMNS = ["shock_gt5", "next_pct_change", "previous_month_change"]


def run_prequential_lightgbm(numeric_features, label):
    all_features = numeric_features + ["geo_id"]
    predictions = []
    origins = sorted(
        d for d in df["price_date"].dropna().unique()
        if FORECAST_START <= pd.Timestamp(d) <= FORECAST_END
    )
    for i, origin in enumerate(origins):
        origin = pd.Timestamp(origin)
        # A row is trainable only once its next-month outcome is known by origin.
        train = df[df["next_date"] <= origin].copy()
        current = df[df["price_date"].eq(origin)].copy()
        if len(train) < 500 or current.empty:
            continue
        x_train = train[all_features].copy()
        x_current = current[all_features].copy()
        x_train["geo_id"] = x_train["geo_id"].astype(str).astype("category")
        categories = x_train["geo_id"].cat.categories
        x_current["geo_id"] = pd.Categorical(
            x_current["geo_id"].astype(str), categories=categories
        )
        model = LGBMClassifier(
            n_estimators=60, num_leaves=12, learning_rate=0.04,
            min_child_samples=35, subsample=0.9, colsample_bytree=0.8,
            reg_lambda=4, class_weight="balanced", verbosity=-1,
            n_jobs=2, random_state=42,
        )
        model.fit(x_train, train["shock_gt5"], categorical_feature=["geo_id"])
        current[f"prob_{label}"] = model.predict_proba(x_current)[:, 1]
        predictions.append(current[[
            "geo_id", "mkt_name", "price_date", "next_date",
            "shock_gt5", "next_pct_change", "previous_month_change", f"prob_{label}"
        ]])
        if i % 12 == 0:
            print(f"{label}: origin {origin:%Y-%m}; train rows={len(train)}; current rows={len(current)}")
    if not predictions:
        raise RuntimeError(f"No prequential predictions generated for {label}.")
    return pd.concat(predictions, ignore_index=True)


def expected_prequential_rows():
    """Return source-derived rows the monthly prediction loop is expected to emit."""
    origins = sorted(
        d for d in df["price_date"].dropna().unique()
        if FORECAST_START <= pd.Timestamp(d) <= FORECAST_END
    )
    eligible_origins = []
    for origin_value in origins:
        origin = pd.Timestamp(origin_value)
        train_count = int((df["next_date"] <= origin).sum())
        current_exists = bool(df["price_date"].eq(origin).any())
        if train_count >= 500 and current_exists:
            eligible_origins.append(origin)
    expected = df[df["price_date"].isin(eligible_origins)][
        CACHE_KEY_COLUMNS + CACHE_CHECK_COLUMNS
    ].copy()
    return expected


def validate_prediction_cache(path, probability_column, expected_rows):
    """Reject cache files that do not match current source keys/labels/derived values."""
    required = set(
        CACHE_KEY_COLUMNS + ["mkt_name"] + CACHE_CHECK_COLUMNS + [probability_column]
    )
    try:
        cache = pd.read_csv(path, parse_dates=["price_date", "next_date"])
    except Exception as exc:
        return False, None, f"could not read cache ({exc})"
    missing = sorted(required - set(cache.columns))
    if missing:
        return False, None, f"missing columns: {missing}"
    cache["price_date"] = pd.to_datetime(cache["price_date"], errors="coerce")
    cache["next_date"] = pd.to_datetime(cache["next_date"], errors="coerce")
    if cache[["price_date", "next_date"]].isna().any().any():
        return False, None, "one or more cached keys have invalid dates"
    if cache.duplicated(CACHE_KEY_COLUMNS).any():
        return False, None, "duplicate market/month keys"

    # Normalize IDs and dates only for validation; leave loaded data intact for modelling.
    cache_check = cache[CACHE_KEY_COLUMNS + CACHE_CHECK_COLUMNS].copy()
    expected_check = expected_rows[CACHE_KEY_COLUMNS + CACHE_CHECK_COLUMNS].copy()
    for frame in (cache_check, expected_check):
        frame["geo_id"] = frame["geo_id"].astype(str)
        frame["price_date"] = pd.to_datetime(frame["price_date"], errors="coerce").dt.normalize()
        frame["next_date"] = pd.to_datetime(frame["next_date"], errors="coerce").dt.normalize()
    try:
        joined = cache_check.merge(
            expected_check,
            on=CACHE_KEY_COLUMNS,
            how="outer",
            suffixes=("_cache", "_source"),
            indicator=True,
            validate="one_to_one",
        )
    except Exception as exc:
        return False, None, f"key comparison failed ({exc})"
    if not joined["_merge"].eq("both").all():
        cache_only = int(joined["_merge"].eq("left_only").sum())
        source_only = int(joined["_merge"].eq("right_only").sum())
        return False, None, f"market/month keys differ (cache-only={cache_only}, source-only={source_only})"

    for col in CACHE_CHECK_COLUMNS:
        left = pd.to_numeric(joined[f"{col}_cache"], errors="coerce").to_numpy(dtype=float)
        right = pd.to_numeric(joined[f"{col}_source"], errors="coerce").to_numpy(dtype=float)
        if not np.isclose(left, right, rtol=1e-7, atol=1e-9, equal_nan=True).all():
            return False, None, f"cached {col} values differ from source-derived values"

    probabilities = pd.to_numeric(cache[probability_column], errors="coerce").to_numpy(dtype=float)
    if not np.isfinite(probabilities).all() or ((probabilities < 0) | (probabilities > 1)).any():
        return False, None, f"invalid values in {probability_column}"
    return True, cache, "keys and source-derived target/baseline values match"


core_cache = Path("marketshield_prequential_core_predictions.csv")
enhanced_cache = Path("marketshield_prequential_enhanced_predictions.csv")
expected_cache_rows = expected_prequential_rows()
use_cached_predictions = False

if not RETRAIN_MODELS and core_cache.exists() and enhanced_cache.exists():
    core_ok, cached_core, core_reason = validate_prediction_cache(
        core_cache, "prob_core", expected_cache_rows
    )
    enhanced_ok, cached_enhanced, enhanced_reason = validate_prediction_cache(
        enhanced_cache, "prob_enhanced", expected_cache_rows
    )
    if core_ok and enhanced_ok:
        print("Validated saved monthly prediction caches:")
        print("  core:", core_reason)
        print("  enhanced:", enhanced_reason)
        preq_core = cached_core
        preq_enhanced = cached_enhanced
        use_cached_predictions = True
    else:
        print("Saved prediction cache rejected; source-derived rows do not match:")
        print("  core:", core_reason)
        print("  enhanced:", enhanced_reason)
        print("Both LightGBM variants will be refit to keep the comparison consistent.")
elif RETRAIN_MODELS:
    print("RETRAIN_MODELS=True: saved prediction caches will be ignored.")
else:
    print("One or both prediction cache files are missing; models will be refit.")

if not use_cached_predictions:
    print("Running core-feature LightGBM for each forecast origin...")
    preq_core = run_prequential_lightgbm(core_numeric, "core")
    print("Running LightGBM with additional data-quality features...")
    preq_enhanced = run_prequential_lightgbm(enhanced_numeric, "enhanced")

join_keys = ["geo_id", "price_date", "next_date"]
preq = preq_enhanced.merge(
    preq_core[join_keys + ["prob_core"]], on=join_keys, how="inner", validate="one_to_one",
    suffixes=("", "_core")
)
if "shock_gt5_core" in preq and not (preq["shock_gt5"] == preq["shock_gt5_core"]).all():
    raise ValueError("Cached/model prediction labels do not match across feature variants.")
if "shock_gt5_core" in preq:
    preq = preq.drop(columns=["shock_gt5_core"])
if len(preq) != len(expected_cache_rows):
    raise ValueError(
        f"Prediction row count {len(preq)} does not match expected source-derived row count "
        f"{len(expected_cache_rows)}. Inspect source coverage and cache validation."
    )
print("Prequential predictions:", preq.shape)
display(preq.head())


Validated saved monthly prediction caches:
  core: keys and source-derived target/baseline values match
  enhanced: keys and source-derived target/baseline values match
Prequential predictions: (2088, 9)


,geo_id,price_date,next_date,mkt_name,shock_gt5,next_pct_change,previous_month_change,prob_enhanced,prob_core
0,gid_126500000540300000,2018-12-01,2019-01-01,Soqatra (Hudaibo),0,0.000000,0.034483,0.639693,0.629303
1,gid_127900000450200000,2018-12-01,2019-01-01,Aden City,0,-0.033816,-0.050459,0.404044,0.412921
2,gid_130600000448800000,2018-12-01,2019-01-01,Al Hawtah- Main road - Back Street,1,0.063231,0.083333,0.404887,0.408827
3,gid_131300000453800000,2018-12-01,2019-01-01,Zungubar City,0,0.022409,-0.088235,0.360317,0.343543
4,gid_135800000440200000,2018-12-01,2019-01-01,Taiz City,1,0.053690,-0.018706,0.376364,0.425866


## 6. Choose the threshold on validation only; report the later period separately

Validation outcomes are target months from January 2019 through December 2021. The retrospective test outcomes are target months from January 2022 through March 2026. The later period is not a completely untouched holdout because prior experiments already inspected aggregated results for overlapping dates.

In [ ]:
# Rebuild the validation and later-period splits in this cell so a fresh kernel
# never depends on variables left over from an earlier notebook execution.
validation = preq[
    (preq["next_date"] >= pd.Timestamp("2019-01-01"))
    & (preq["next_date"] < pd.Timestamp("2022-01-01"))
].copy()
test = preq[preq["next_date"] >= pd.Timestamp("2022-01-01")].copy()

if validation.empty or test.empty:
    raise ValueError("Validation or later-period split is empty; inspect source date coverage.")

# Select each decision threshold on the validation period only.
threshold_rows = []
selected_thresholds = {}
for threshold_key, prob_col in [("core", "prob_core"), ("enhanced", "prob_enhanced")]:
    y_validation = validation["shock_gt5"].to_numpy(dtype=int)
    for threshold in np.arange(0.10, 0.801, 0.025):
        pred_validation = (validation[prob_col].to_numpy(dtype=float) >= threshold).astype(int)
        threshold_rows.append({
            "model": threshold_key,
            "threshold": float(threshold),
            "precision": precision_score(y_validation, pred_validation, zero_division=0),
            "recall": recall_score(y_validation, pred_validation, zero_division=0),
            "f1": f1_score(y_validation, pred_validation, zero_division=0),
            "alerts": int(pred_validation.sum()),
        })

threshold_table = pd.DataFrame(threshold_rows)
for threshold_key in ("core", "enhanced"):
    candidates = threshold_table[threshold_table["model"].eq(threshold_key)]
    best = candidates.sort_values(["f1", "precision"], ascending=False).iloc[0]
    selected_thresholds[threshold_key] = float(best["threshold"])

print("Validation rows:", len(validation), "| Later-period replay rows:", len(test))
print("Positive rate — validation:", round(float(validation["shock_gt5"].mean()), 3))
print("Positive rate — later-period replay:", round(float(test["shock_gt5"].mean()), 3))
print("Thresholds selected using validation only:", selected_thresholds)

def classification_report_row(model, split, frame, prob_col, threshold):
    y = frame["shock_gt5"].to_numpy(dtype=int)
    prob = frame[prob_col].to_numpy(dtype=float)
    pred = (prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    alert_count = int(pred.sum())
    return {
        "model": model,
        "split": split,
        "threshold": round(float(threshold), 3),
        "n": int(len(frame)),
        "positive_rate": float(np.mean(y)),
        "roc_auc": float(roc_auc_score(y, prob)),
        "average_precision": float(average_precision_score(y, prob)),
        "precision": float(precision_score(y, pred, zero_division=0)),
        "recall": float(recall_score(y, pred, zero_division=0)),
        "f1": float(f1_score(y, pred, zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y, pred)),
        "alerts": alert_count,
        "TP": int(tp),
        "FP": int(fp),
        "FN": int(fn),
        "TN": int(tn),
        "false_alert_share": float(fp / alert_count) if alert_count else 0.0,
    }

summary_columns = [
    "model", "split", "threshold", "n", "positive_rate", "roc_auc",
    "average_precision", "precision", "recall", "f1", "balanced_accuracy",
    "alerts", "TP", "FP", "FN", "TN", "false_alert_share",
]
summary_rows = []
for model_key, prob_col, threshold_key in [
    ("core", "prob_core", "core"),
    ("expanded_quality", "prob_enhanced", "enhanced"),
]:
    threshold = selected_thresholds[threshold_key]
    for split_name, frame in [
        ("validation", validation),
        ("later_period_replay", test),
    ]:
        summary_rows.append(
            classification_report_row(model_key, split_name, frame, prob_col, threshold)
        )
summary = pd.DataFrame(summary_rows, columns=summary_columns)
display(summary.round(3))

# Interpretable momentum rule: flag when current month's c_wheat rose >5% from its prior month.
rule_summary = []
for split_name, frame in [("Momentum rule — validation", validation), ("Momentum rule — later-period replay", test)]:
    y = frame["shock_gt5"].to_numpy(dtype=int)
    pred = (frame["previous_month_change"].to_numpy(dtype=float) > 0.05).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    rule_summary.append({
        "Model / rule": split_name,
        "Precision": precision_score(y, pred, zero_division=0),
        "Recall": recall_score(y, pred, zero_division=0),
        "F1": f1_score(y, pred, zero_division=0),
        "Alerts": int(pred.sum()), "TP": int(tp), "FP": int(fp),
        "FN": int(fn), "TN": int(tn),
    })
display(pd.DataFrame(rule_summary).round(3))

# Force LF line endings so CSV artifacts remain stable when running on Windows.
preq.to_csv("marketshield_prequential_predictions.csv", index=False, lineterminator="\n")
summary.to_csv("marketshield_prequential_summary.csv", index=False, lineterminator="\n")
print("Saved marketshield_prequential_predictions.csv and marketshield_prequential_summary.csv")


## 7. Interpretation and operational-readiness gate

### Current result: research prototype only

Two LightGBM variants are compared with the same monthly prequential protocol, target and time windows. The **core** model uses price lags, log changes and seasonality; the **expanded** model adds data-quality indicators. Thresholds are selected by F1 on the 2019–2021 validation predictions and then frozen for the 2022–March 2026 replay.

**Use the metric tables emitted by Section 6 for the results of the current execution.** The numerical results are intentionally not duplicated in this text, because an incompatible prediction cache can be rejected and the models refit; in that case, previously written figures could become stale. Review precision, recall, F1, false-positive counts, and the momentum-rule comparison from the actual output before reporting results.

A high recall obtained by issuing many false alerts may not provide operational value. The greater-than-5% target is a narrow price-change label, not a direct measure of a food crisis, household hardship, or humanitarian need.

### Why the later period is not fully independent

The 2022–March 2026 dates were inspected in earlier project experiments. This monthly replay limits target-time leakage, but it does not undo earlier inspection or the potential revision of historical source estimates. These results are a **retrospective walk-forward replay**, not an untouched independent test. Saved predictions are reused only after their keys, labels, target changes, and momentum-baseline values are checked against the current source-derived rows. Set `RETRAIN_MODELS = True` to force both models to be refit.

### What is required before any live use

1. Obtain point-in-time snapshots or independent market observations with collection dates; this source release ends in March 2026, and historical estimates can be revised.
2. Freeze models and thresholds before evaluating on a future, uninspected period or independent observed prices.
3. Predefine the alert objective and the costs of false alarms and missed events.
4. Report performance by market and period, confidence intervals, probability calibration, and data-quality failure cases.
5. Run in shadow mode with human review and no public/humanitarian alerts before considering any pilot.

**Conclusion:** keep Yemen MarketShield AI at research-prototype status. The label is a greater-than-5% rise in the next month's `c_wheat` close estimate; it is not an official food crisis or humanitarian outcome. Do not use predictions for operational decisions.

### External source discovery

The Yemen Ministry of Industry, Trade and Investment publishes weekly price-list observations for wheat grain and wheat flour (for example, the release dated 17 September 2026), but the retrieved table does not match this model's per-market monthly `c_wheat` target. The WFP food-price database documented by HDX HAPI has monthly price series, market identifiers, price type, unit, currency, and reference period fields. It is the next candidate to inspect for an independent holdout, but its coverage after March 2026 and overlap with the World Bank estimates' upstream data must be confirmed before calling it independent. API use requires an application identifier. See https://moit.gov.ye/complaints/prices and https://hdx-hapi.readthedocs.io/en/latest/data_usage_guides/food_security_nutrition_and_poverty/.
